In [1]:
import polars as pl
import numpy as np
import pickle
import itertools
from joblib import Parallel, delayed

In [7]:
with open("data/pref_distances.pickle", 'rb') as file:
    furthest_ranks = pickle.load(file)

len(furthest_ranks)

46

In [3]:
sushi_path = "./data/sushi3.idata"
sushi_df = pl.read_csv(
    sushi_path, separator="\t", has_header=False, 
    new_columns=[
        "id", "name", "style", "major_group",
        "minor_group", "oiliness", "eaten_frequency",
        "price", "sold_frequency"
    ]
)

sushi_df.tail()

id,name,style,major_group,minor_group,oiliness,eaten_frequency,price,sold_frequency
i64,str,i64,i64,i64,f64,f64,f64,f64
95,"""karei""",1,0,2,2.6,1.094737,1.0,0.04
96,"""hiramasa""",1,0,0,1.970588,1.0,1.0,0.04
97,"""namako""",1,0,8,1.936709,0.443038,1.5,0.04
98,"""shishamo""",1,0,0,2.16,0.613333,1.0,0.04
99,"""kaki""",1,0,4,1.779221,0.727273,1.25,0.04


In [4]:
def prep_sushi(df, nominal_fields, numerical_fields, drop_fields, num_rows, id_field = "id"):
    df = df.limit(
        num_rows
    ).sort(
        id_field
    )
    arr = df.drop(
        drop_fields
    ).with_columns(
        (pl.col(numerical_fields) - pl.col(numerical_fields).min()) / (pl.col(numerical_fields).max() - pl.col(numerical_fields).min())
    ).with_columns(
        df[nominal_fields].to_dummies(drop_first=True)
    ).to_numpy()

    inds = (df[id_field].sort() + 1).to_list()

    return arr, inds

nominal_fields = ["minor_group"]
numerical_fields = ["oiliness", "eaten_frequency", "price", "sold_frequency"]
drop_fields = ["id", "name", "major_group"] + nominal_fields

sushi_arr, sushi_inds = prep_sushi(sushi_df, nominal_fields, numerical_fields, drop_fields, 10)

sushi_arr

array([[1.        , 0.85828263, 0.70201946, 0.2334073 , 0.5       ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        ],
       [1.        , 0.14765004, 0.49182399, 0.2780176 , 0.75      ,
        0.        , 1.        , 0.        , 0.        , 0.        ,
        0.        ],
       [1.        , 0.48005296, 1.        , 0.24392127, 0.75      ,
        1.        , 0.        , 0.        , 0.        , 0.        ,
        0.        ],
       [1.        , 0.84228598, 0.56701463, 0.13978737, 1.        ,
        0.        , 0.        , 1.        , 0.        , 0.        ,
        0.        ],
       [1.        , 0.1029679 , 0.        , 0.65300401, 0.75      ,
        0.        , 0.        , 0.        , 0.        , 1.        ,
        0.        ],
       [1.        , 1.        , 0.10477308, 0.10190094, 0.        ,
        0.        , 0.        , 1.        , 0.        , 0.        ,
        0.        ],
       [1.        , 0.28109149, 0.47655331, 0.48158165, 0.

In [8]:
distances = list(int(dist) for dist in furthest_ranks)
human_utility_flags = [True, False]
human_betas = [1.5, 0.5]
algorithm_betas = [1.5]
algorithm_alphas = [1.5, 0.0]
algorithm_sigmas = [1.5]

p_penalty = 0.321
algo_k = 2
universe = [0] + sushi_inds

NameError: name 'sushi_inds' is not defined

In [7]:
len(furthest_ranks[0])

80

In [7]:
# def process_pair(human_ranks, algo_ranks):
#     utility_arr = np.zeros(11)
#     if use_human_utility:
#         utility_arr[human_ranks[0]] = 1
#     else:
#         utility_arr[algo_ranks[0]] = 1

#     human = TopKMallows(center=human_ranks, k=10, beta=human_beta, p=p_penalty)
#     algo = DiverseTopKMallows(center=algo_ranks, k=algo_k, beta=algo_beta, p=p_penalty, embeddings=sushi_arr, alpha=algo_alpha, sigma=algo_sigma)

#     util = human.collab_utility(algo, universe, utility_arr)

#     return human_ranks, algo_ranks, util

# pairs = furthest_ranks[0]

# res = Parallel(n_jobs=-2, verbose=10)(
#     delayed(process_pair)(*pair) for pair in pairs
# )

In [8]:
universe

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [16]:
if None in furthest_ranks:
    del furthest_ranks[None]

In [17]:
len(furthest_ranks)

46

In [10]:
list(
    itertools.product(
        distances, human_utility_flags, human_betas, algorithm_betas, 
        algorithm_alphas, algorithm_sigmas
    )
)

[(21, True, 1.5, 1.5, 1.5, 1.5),
 (21, True, 1.5, 1.5, 0.0, 1.5),
 (21, True, 0.5, 1.5, 1.5, 1.5),
 (21, True, 0.5, 1.5, 0.0, 1.5),
 (21, False, 1.5, 1.5, 1.5, 1.5),
 (21, False, 1.5, 1.5, 0.0, 1.5),
 (21, False, 0.5, 1.5, 1.5, 1.5),
 (21, False, 0.5, 1.5, 0.0, 1.5),
 (25, True, 1.5, 1.5, 1.5, 1.5),
 (25, True, 1.5, 1.5, 0.0, 1.5),
 (25, True, 0.5, 1.5, 1.5, 1.5),
 (25, True, 0.5, 1.5, 0.0, 1.5),
 (25, False, 1.5, 1.5, 1.5, 1.5),
 (25, False, 1.5, 1.5, 0.0, 1.5),
 (25, False, 0.5, 1.5, 1.5, 1.5),
 (25, False, 0.5, 1.5, 0.0, 1.5),
 (18, True, 1.5, 1.5, 1.5, 1.5),
 (18, True, 1.5, 1.5, 0.0, 1.5),
 (18, True, 0.5, 1.5, 1.5, 1.5),
 (18, True, 0.5, 1.5, 0.0, 1.5),
 (18, False, 1.5, 1.5, 1.5, 1.5),
 (18, False, 1.5, 1.5, 0.0, 1.5),
 (18, False, 0.5, 1.5, 1.5, 1.5),
 (18, False, 0.5, 1.5, 0.0, 1.5),
 (16, True, 1.5, 1.5, 1.5, 1.5),
 (16, True, 1.5, 1.5, 0.0, 1.5),
 (16, True, 0.5, 1.5, 1.5, 1.5),
 (16, True, 0.5, 1.5, 0.0, 1.5),
 (16, False, 1.5, 1.5, 1.5, 1.5),
 (16, False, 1.5, 1.5, 0.0, 1.

In [11]:
from mallows import TopKMallows, DiverseTopKMallows

def process_combo(distance: int, use_human_utility: bool, human_beta: float, algo_beta: float, algo_alpha: float, algo_sigma: float):
    def process_pair(human_ranks, algo_ranks):
        utility_arr = np.zeros(11)
        if use_human_utility:
            utility_arr[human_ranks[0]] = 1
        else:
            utility_arr[algo_ranks[0]] = 1

        human = TopKMallows(center=human_ranks, k=10, beta=human_beta, p=p_penalty)
        algo = DiverseTopKMallows(center=algo_ranks, k=algo_k, beta=algo_beta, p=p_penalty, embeddings=sushi_arr, alpha=algo_alpha, sigma=algo_sigma)

        #print(human_ranks, algo_ranks, universe)
        util = human.collab_utility(algo, universe, utility_arr)

        return human_ranks, algo_ranks, util

    pairs = furthest_ranks[distance]

    pair_res = Parallel(n_jobs=-2, verbose=10)(
        delayed(process_pair)(*pair) for pair in pairs
    )

    return pair_res, distance, use_human_utility, human_beta, algo_beta, algo_alpha, algo_sigma

parameter_combos = list(itertools.product(distances, human_utility_flags, human_betas, algorithm_betas, algorithm_alphas, algorithm_sigmas))

res = Parallel(n_jobs=-2, verbose=10)(
    delayed(process_combo)(*combo) for combo in parameter_combos
)

[Parallel(n_jobs=-2)]: Using backend LokyBackend with 11 concurrent workers.
[Parallel(n_jobs=-2)]: Using backend ThreadingBackend with 11 concurrent workers.
[Parallel(n_jobs=-2)]: Done   3 tasks      | elapsed:    1.5s
[Parallel(n_jobs=-2)]: Done  10 tasks      | elapsed:    1.6s
[Parallel(n_jobs=-2)]: Done  19 tasks      | elapsed:    2.1s
[Parallel(n_jobs=-2)]: Done  28 tasks      | elapsed:    2.5s
[Parallel(n_jobs=-2)]: Done  39 tasks      | elapsed:    3.0s
[Parallel(n_jobs=-2)]: Done  50 tasks      | elapsed:    3.4s
[Parallel(n_jobs=-2)]: Done  63 tasks      | elapsed:    3.9s
[Parallel(n_jobs=-2)]: Done  76 tasks      | elapsed:    4.4s
[Parallel(n_jobs=-2)]: Done  91 tasks      | elapsed:    5.4s
[Parallel(n_jobs=-2)]: Done 106 tasks      | elapsed:    6.6s
[Parallel(n_jobs=-2)]: Done 123 tasks      | elapsed:    8.4s
[Parallel(n_jobs=-2)]: Done 140 tasks      | elapsed:    9.8s
[Parallel(n_jobs=-2)]: Done 159 tasks      | elapsed:   12.0s
[Parallel(n_jobs=-2)]: Done 178 tas

: 

In [10]:
data = []
for item in res:
    rank_pairs, *rest = item
    for pair in rank_pairs:
        data.append((*pair, *rest))

df = pl.DataFrame(data, orient="row", schema=[
    "Human-Ranking", "Algo-Ranking", "Utility", "Distance", "Uses-Human-Utility", "Human-Beta", "Algorithm-Beta", "Algorithm-Alpha", "Algorithm-Sigma"
])

df

Human-Ranking,Algo-Ranking,Utility,Distance,Uses-Human-Utility,Human-Beta,Algorithm-Beta,Algorithm-Alpha,Algorithm-Sigma
list[i64],list[i64],f64,i64,bool,f64,f64,f64,f64
"[6, 1, … 3]","[1, 10, … 5]",0.169857,21,true,1.5,1.5,1.5,1.5
"[6, 1, … 3]","[8, 5, … 7]",0.102781,21,true,1.5,1.5,1.5,1.5
"[6, 1, … 3]","[1, 10, … 5]",0.075243,21,true,1.5,1.5,0.0,1.5
"[6, 1, … 3]","[8, 5, … 7]",0.087443,21,true,1.5,1.5,0.0,1.5
"[6, 1, … 3]","[1, 10, … 5]",0.134605,21,true,0.5,1.5,1.5,1.5
…,…,…,…,…,…,…,…,…
"[6, 1, … 3]","[6, 1, … 7]",0.799351,15,true,1.5,1.5,0.0,1.5
"[6, 1, … 3]","[8, 4, … 3]",0.098713,15,true,0.5,1.5,1.5,1.5
"[6, 1, … 3]","[6, 1, … 7]",0.679108,15,true,0.5,1.5,1.5,1.5


ShapeError: data does not match the number of columns

In [28]:
import random

data = [
    ([random.randint(1, 1000), 7, 8, 2, 0, 3, 6, 9, 5, 4], [random.randint(1, 1000), 2, 0, 1, 8, 3, 6, 9, 5, 4], np.float64(0.1901257889260058), random.randint(23, 24), True, 1.5, 1.5, 0.85, 1.5)
    for _ in range(1000)
]

schema = [
    "Human-Ranking", "Algo-Ranking", "Utility", "Distance", "Uses-Human-Utility", "Human-Beta", "Algorithm-Beta", "Algorithm-Alpha", "Algorithm-Sigma"
]

df = pl.DataFrame(data, orient="row", schema=schema)

df

Human-Ranking,Algo-Ranking,Utility,Distance,Uses-Human-Utility,Human-Beta,Algorithm-Beta,Algorithm-Alpha,Algorithm-Sigma
list[i64],list[i64],f64,i64,bool,f64,f64,f64,f64
"[683, 7, … 4]","[545, 2, … 4]",0.190126,24,true,1.5,1.5,0.85,1.5
"[89, 7, … 4]","[657, 2, … 4]",0.190126,23,true,1.5,1.5,0.85,1.5
"[729, 7, … 4]","[964, 2, … 4]",0.190126,24,true,1.5,1.5,0.85,1.5
"[675, 7, … 4]","[379, 2, … 4]",0.190126,23,true,1.5,1.5,0.85,1.5
"[987, 7, … 4]","[422, 2, … 4]",0.190126,24,true,1.5,1.5,0.85,1.5
…,…,…,…,…,…,…,…,…
"[57, 7, … 4]","[840, 2, … 4]",0.190126,23,true,1.5,1.5,0.85,1.5
"[274, 7, … 4]","[134, 2, … 4]",0.190126,23,true,1.5,1.5,0.85,1.5
"[141, 7, … 4]","[876, 2, … 4]",0.190126,23,true,1.5,1.5,0.85,1.5


In [37]:
#import pyarrow.dataset as ds
import uuid

def path_provider(args: pl.FileProviderArgs):
    assert args.index_in_partition == 0

    return f"Distance_{args.partition_keys.cast(pl.String).item()}/{uuid.uuid4().hex}.parquet"

chunk_size = 499
chunk = []
for item in data:
    chunk.append(item)
    if len(chunk) >= chunk_size:
        df = pl.LazyFrame(chunk, orient="row", schema=schema)
        df.sink_parquet(pl.PartitionBy(
            "./data/diversity_utilities",
            key="Distance",
            file_path_provider=path_provider
        ))
        # ds.write_dataset(
        #     df.to_arrow(),
        #     base_dir="./data/diversity_utilities",
        #     format="parquet",
        #     partitioning=["Distance"],
        #     partitioning_flavor="hive",
        #     existing_data_behavior="overwrite_or_ignore"
        # )
        chunk.clear()

if len(chunk) > 0:
    df = pl.LazyFrame(chunk, orient="row", schema=schema)
    df.sink_parquet(pl.PartitionBy(
        "./data/diversity_utilities",
        key="Distance",
        file_path_provider=path_provider
    ))
    # ds.write_dataset(
    #     df.to_arrow(),
    #     base_dir="./data/diversity_utilities",
    #     format="parquet",
    #     partitioning=["Distance"],
    #     partitioning_flavor="hive",
    #     existing_data_behavior="overwrite_or_ignore"
    # )
    del chunk

In [13]:
df.write_parquet("./data/diversity_utilities", partition_by=["Distance"])

In [2]:
df = pl.read_parquet("./data/diversity_utilities")
df

Human-Ranking,Algo-Ranking,Utility,Distance,Uses-Human-Utility,Human-Beta,Algorithm-Beta,Algorithm-Alpha,Algorithm-Sigma
list[i64],list[i64],f64,i64,bool,f64,f64,f64,f64
"[5, 8, … 10]","[5, 8, … 10]",0.782676,0,true,1.5,1.5,1.5,1.5
"[8, 3, … 10]","[8, 3, … 10]",0.776962,0,true,1.5,1.5,1.5,1.5
"[8, 3, … 7]","[8, 3, … 7]",0.776905,0,true,1.5,1.5,1.5,1.5
"[8, 3, … 7]","[8, 3, … 7]",0.776905,0,true,1.5,1.5,1.5,1.5
"[2, 8, … 10]","[2, 8, … 10]",0.776727,0,true,1.5,1.5,1.5,1.5
…,…,…,…,…,…,…,…,…
"[6, 4, … 1]","[6, 8, … 2]",0.673619,9,false,0.5,1.5,1.5,1.5
"[6, 4, … 1]","[6, 8, … 5]",0.673619,9,false,0.5,1.5,1.5,1.5
"[6, 4, … 1]","[8, 6, … 5]",0.53873,9,false,0.5,1.5,1.5,1.5


In [3]:
distance_counts = df["Distance"].value_counts()

with pl.Config(tbl_rows=-1):
    print(distance_counts)

shape: (6, 2)
┌──────────┬─────────┐
│ Distance ┆ count   │
│ ---      ┆ ---     │
│ i64      ┆ u32     │
╞══════════╪═════════╡
│ 45       ┆ 32      │
│ 0        ┆ 640     │
│ 27       ┆ 2854704 │
│ 18       ┆ 6214392 │
│ 9        ┆ 1855496 │
│ 36       ┆ 201888  │
└──────────┴─────────┘
